# 02 · Discount cadence
Frequency, depth and rhythm of discounting, per product, weekday and brand.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from pricing.db import get_engine, load_snapshots, load_scrape_log
from pricing.clean import clean_snapshots, dim_products, gap_report
from pricing import analysis as an
from pricing.export import load_matches
raw = load_snapshots(get_engine())
clean, report = clean_snapshots(raw)
events = an.discount_events(clean)
print(f"{len(raw):,} raw rows -> {len(clean):,} clean rows, {clean.product_url.nunique()} products")

In [ ]:
cad = an.product_cadence(events, clean)
cad.describe().T

In [ ]:
daily = clean.assign(day=clean.scraped_at.dt.tz_convert(an.IST).dt.date).groupby(["platform", "day"]).discount_pct.mean().unstack(0)
daily.plot(figsize=(9, 3.5), title="Average discount by platform (%)");

In [ ]:
pd.DataFrame(an.weekday_pattern(events)).set_index("weekday").plot.bar(figsize=(7, 3), legend=False, title="Deeper-discount events by weekday");

In [ ]:
an.brand_typology(clean, events).sort_values("avg_discount", ascending=False)

In [ ]:
matches = load_matches(Path("../data/products.csv")) if (Path("../data/products.csv")).exists() else pd.DataFrame(columns=["product_url","match_id"])
an.first_mover(events, matches)